# Official HEC-HMS Guide Mirror: Basic Model Setup

Official guide: https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/basic-model-setup

This notebook mirrors the Basic Model Setup guide at a programmatic level: extract an HMS sample project, inspect core components, select a run, compute it headlessly, and confirm the expected HMS output files exist. It uses the HMS `castro` sample project through `HmsExamples.extract_project()` so the workflow stays reproducible.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import HmsCmdr

project, project_path = init_sample_project("castro", "22_basic_model_setup")
run_name = "Current"

component_counts = pd.DataFrame([
    {"component": "Basin models", "count": len(project.basin_df)},
    {"component": "Meteorologic models", "count": len(project.met_df)},
    {"component": "Control specs", "count": len(project.control_df)},
    {"component": "Simulation runs", "count": len(project.run_df)},
    {"component": "Time-series gages", "count": len(project.gage_df)},
])
assert component_counts["count"].sum() > 0
component_counts

,component,count
0,Basin models,2
1,Meteorologic models,1
2,Control specs,1
3,Simulation runs,2
4,Time-series gages,2


In [3]:
config = project.get_run_configuration(run_name)
run_summary = pd.DataFrame([{
    "run_name": config["run_name"],
    "basin_model": config["basin_name"],
    "met_model": config["met_name"],
    "control_spec": config["control_name"],
    "output_dss": config["dss_file"],
    "basin_area_sqmi": round(float(config["basin_area"]), 2),
    "control_start": config["control_start"],
    "control_end": config["control_end"],
    "interval_minutes": int(config["control_interval_minutes"]),
}])
run_summary

,run_name,basin_model,met_model,control_spec,output_dss,basin_area_sqmi,control_start,control_end,interval_minutes
0,Current,Castro 1,GageWts,Jan73,Current.dss,40.51,1973-01-16 03:00:00,1973-01-16 12:55:00,5


In [4]:
success = HmsCmdr.compute_run(
    run_name,
    hms_object=project,
    timeout=120,
    save_project=False,
    max_memory="2G",
)
assert success, f"HMS compute failed for run {run_name}"

expected_outputs = [
    project_path / config["dss_file"],
    project_path / f"{run_name}.log",
    project_path / f"{run_name}.out",
]
output_check = pd.DataFrame([
    {
        "file": path.name,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else 0.0,
    }
    for path in expected_outputs
])
assert output_check.loc[output_check["file"] == config["dss_file"], "exists"].iloc[0]
output_check

,file,exists,size_kb
0,Current.dss,True,267.0
1,Current.log,True,2.6
2,Current.out,True,920.8


## Coverage Notes

The official GUI guide introduces the same project concepts: basin model, meteorologic model, control specification, simulation run, compute, and results review. This starter mirror validates the hms-commander equivalent without reading DSS content, keeping it independent of the optional DSS stack tracked outside CLB-238.